In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from scipy import stats
from scipy.stats import ttest_ind, chi2_contingency, pearsonr


from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [7]:
df=pd.read_csv('Clean_Dataset.csv')
df.head()

,Unnamed: 0,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price
0,0,SpiceJet,SG-8709,Delhi,Evening,zero,Night,Mumbai,Economy,2.17,1,5953
1,1,SpiceJet,SG-8157,Delhi,Early_Morning,zero,Morning,Mumbai,Economy,2.33,1,5953
2,2,AirAsia,I5-764,Delhi,Early_Morning,zero,Early_Morning,Mumbai,Economy,2.17,1,5956
3,3,Vistara,UK-995,Delhi,Morning,zero,Afternoon,Mumbai,Economy,2.25,1,5955
4,4,Vistara,UK-963,Delhi,Morning,zero,Morning,Mumbai,Economy,2.33,1,5955


In [8]:
df = df.drop(columns=["flight"])

In [9]:
X = df.drop(columns=["price"])
y = df["price"]


In [10]:
X = pd.get_dummies(X, columns=[
    "airline",
    "source_city",
    "destination_city",
    "departure_time",
    "arrival_time"
])
X = X.apply(lambda x: x.astype(int) if x.dtype == 'bool' else x)

In [11]:
X['stops'] = X['stops'].map({
    "zero": 0,
    "one": 1,
    "two_or_more": 2
})



In [12]:
X['class'] = X['class'].map({
    "Economy": 0,
    "Business": 1
})

In [13]:
# standarisation
scaler = StandardScaler()

X[["duration", "days_left"]] = scaler.fit_transform(X[["duration", "days_left"]])

In [14]:
display(X.head())

,Unnamed: 0,stops,class,duration,days_left,airline_AirAsia,airline_Air_India,airline_GO_FIRST,airline_Indigo,airline_SpiceJet,...,departure_time_Evening,departure_time_Late_Night,departure_time_Morning,departure_time_Night,arrival_time_Afternoon,arrival_time_Early_Morning,arrival_time_Evening,arrival_time_Late_Night,arrival_time_Morning,arrival_time_Night
0,0,0,0,-1.397531,-1.843875,0,0,0,0,1,...,1,0,0,0,0,0,0,0,0,1
1,1,0,0,-1.375284,-1.843875,0,0,0,0,1,...,0,0,0,0,0,0,0,0,1,0
2,2,0,0,-1.397531,-1.843875,1,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,3,0,0,-1.386407,-1.843875,0,0,0,0,0,...,0,0,1,0,1,0,0,0,0,0
4,4,0,0,-1.375284,-1.843875,0,0,0,0,0,...,0,0,1,0,0,0,0,0,1,0


In [15]:
#splite test-trainig
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [16]:
# create dummy model
model = DummyRegressor(strategy="mean")

# train model
model.fit(X_train, y_train)

# predictions
y_pred = model.predict(X_test)

# evaluation
print("MSE:", mean_squared_error(y_test, y_pred))
print("R2 Score:", r2_score(y_test, y_pred))

MSE: 515482303.1678328
R2 Score: -5.741993791552602e-08


In [17]:
# create model
ridge = Ridge(alpha=1.0)

# train model
ridge.fit(X_train, y_train)

# predictions
y_pred = ridge.predict(X_test)

# evaluation
print("MSE:", mean_squared_error(y_test, y_pred))
print("R2 Score:", r2_score(y_test, y_pred))

MSE: 46074935.06916094
R2 Score: 0.9106178089303298


In [18]:
# create model
rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

# train
rf.fit(X_train, y_train)

# predict
y_pred = rf.predict(X_test)

# evaluation
print("MSE:", mean_squared_error(y_test, y_pred))
print("R2 Score:", r2_score(y_test, y_pred))

MSE: 6184461.633979639
R2 Score: 0.988002571667184


##### Après avoir testé DummyRegressor, Ridge Regression et Random Forest Regressor, le modèle Random Forest Regressor a donné les meilleurs résultats. Il a l’erreur la plus faible (MSE = 6 184 461.63) et le meilleur score (R² = 0.988), ce qui montre une très bonne précision de prédiction. Il a donc été choisi comme modèle final.

In [19]:
import numpy as np
from sklearn.metrics import mean_absolute_error

# predictions
y_pred = rf.predict(X_test)

# errors
errors = np.abs(y_test - y_pred)

# MAE
mae = mean_absolute_error(y_test, y_pred)

# standard error
std_error = stats.sem(errors)

# 95% confidence interval (simple approximation)
margin = 0.95 * std_error

lower = mae - margin
upper = mae + margin

print("MAE:", mae)
print("95% Confidence Interval:", (lower, upper))

MAE: 950.2172609151938
95% Confidence Interval: (np.float64(941.306391451374), np.float64(959.1281303790137))
